# Phase 4 — Baseline Model

## Goal

Build a simple TF-IDF + Logistic Regression classifier for HumAID.

This model serves as the traditional machine-learning baseline against which later LLM-based approaches will be compared.

The data comes from the cleaned, deduplicated, and stratified splits produced in Phase 3.

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_dir = project_root / "data" / "processed"

train_df = pd.read_csv(processed_dir / "train.csv")
val_df = pd.read_csv(processed_dir / "val.csv")
test_df = pd.read_csv(processed_dir / "test.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_train = vectorizer.fit_transform(train_df["text_clean"])
X_val = vectorizer.transform(val_df["text_clean"])
X_test = vectorizer.transform(test_df["text_clean"])

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

In [ ]:
y_train = train_df["class_label"]
y_val = val_df["class_label"]
y_test = test_df["class_label"]

print("Training labels:", y_train.shape)
print("Validation labels:", y_val.shape)
print("Test labels:", y_test.shape)

print("\nClasses:")
print(y_train.unique())

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

clf.fit(X_train, y_train)

print("Baseline model trained successfully!")

In [ ]:
from sklearn.metrics import classification_report

val_preds = clf.predict(X_val)

print(classification_report(y_val, val_preds))

In [ ]:
from sklearn.metrics import f1_score

macro_f1 = f1_score(y_val, val_preds, average="macro")
weighted_f1 = f1_score(y_val, val_preds, average="weighted")

print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

In [ ]:
import pandas as pd

wrong = val_df[val_preds != y_val]

print("Total validation errors:", len(wrong))

print("\nExamples of misclassified tweets:")
print(
    wrong[["text_clean", "class_label"]]
    .sample(5, random_state=42)
    .to_string(index=False)
)

In [ ]:
import csv

results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

row = {
    "model": "baseline_tfidf_logreg",
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
}

metrics_path = results_dir / "baseline_validation_metrics.csv"
write_header = not metrics_path.exists()

with metrics_path.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if write_header:
        writer.writeheader()
    writer.writerow(row)

print("Saved baseline metrics to:", metrics_path)


In [ ]:
import sys

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.baseline import train_baseline

clf_test, vectorizer_test, X_val_test = train_baseline(train_df, val_df)

print("Baseline module works!")
print("Number of features:", X_val_test.shape[1])
